In [0]:
%run "../00.common/01.environment-config.py"

In [0]:
# COMMAND ----------
 
# MAGIC %run "../00_common/01_environment_config"
 
 
# COMMAND ----------
 
# ============================================================
# SOURCE AND TARGET PATHS
# ============================================================
 
bronze_path = f"{s3_root_path}/Bronze/Nisarga/drivers"
silver_path = f"{s3_root_path}/Silver/Nisarga/drivers"
 
print("Bronze Path :", bronze_path)
print("Silver Path :", silver_path)
 
 
# COMMAND ----------
 
# ============================================================
# STEP 1 - READ BRONZE DRIVERS DATA
# ============================================================
 
drivers_df = (
    spark.read
    .format("delta")
    .load(bronze_path)
)
 
display(drivers_df)
 
 
# COMMAND ----------
 
from pyspark.sql import functions as F
 
 
# COMMAND ----------
 
# ============================================================
# STEP 2 - SELECT REQUIRED COLUMNS AND FLATTEN NAME
# ============================================================
# name is a struct containing givenName and familyName.
# URL is excluded because it is not required for analytics.
 
drivers_selected_df = drivers_df.select(
    F.col("driverId"),
    F.col("name.givenName").alias("given_name"),
    F.col("name.familyName").alias("family_name"),
    F.col("dateOfBirth"),
    F.col("nationality"),
    F.col("ingestion_timestamp"),
    F.col("source_file")
)
 
display(drivers_selected_df)
 
 
# COMMAND ----------
 
# ============================================================
# STEP 3 - STANDARDIZE COLUMN NAMES
# ============================================================
 
drivers_renamed_df = (
    drivers_selected_df
    .withColumnsRenamed({
        "driverId": "driver_id",
        "dateOfBirth": "date_of_birth"
    })
)
 
display(drivers_renamed_df)
 
 
# COMMAND ----------
 
# ============================================================
# STEP 4 - REMOVE NULL BUSINESS KEYS
# ============================================================
 
drivers_valid_df = (
    drivers_renamed_df
    .filter(F.col("driver_id").isNotNull())
)
 
display(drivers_valid_df)
 
 
# COMMAND ----------
 
# ============================================================
# STEP 5 - REMOVE DUPLICATES
# ============================================================
 
drivers_distinct_df = (
    drivers_valid_df
    .dropDuplicates(["driver_id"])
)
 
display(drivers_distinct_df)
 
 
# COMMAND ----------
 
# ============================================================
# STEP 6 - STANDARDIZE TEXT VALUES
# ============================================================
 
drivers_clean_df = (
    drivers_distinct_df
    .withColumn(
        "given_name",
        F.initcap(F.trim(F.col("given_name")))
    )
    .withColumn(
        "family_name",
        F.initcap(F.trim(F.col("family_name")))
    )
    .withColumn(
        "nationality",
        F.initcap(F.trim(F.col("nationality")))
    )
)
 
display(drivers_clean_df)
 
 
# COMMAND ----------
 
# ============================================================
# STEP 7 - CREATE FULL NAME COLUMN
# ============================================================
 
drivers_final_df = (
    drivers_clean_df
    .withColumn(
        "full_name",
        F.concat_ws(
            " ",
            F.col("given_name"),
            F.col("family_name")
        )
    )
)
 
display(drivers_final_df)
 
 
# COMMAND ----------
 
# ============================================================
# STEP 8 - WRITE TO SILVER
# ============================================================
 
(
    drivers_final_df
    .write
    .format("delta")
    .mode("overwrite")
    .save(silver_path)
)
 
 
# COMMAND ----------
 
# ============================================================
# STEP 9 - VALIDATE SILVER DATA
# ============================================================
 
silver_drivers_df = (
    spark.read
    .format("delta")
    .load(silver_path)
)
 
display(silver_drivers_df)
 
print("Bronze Records :", drivers_df.count())
print("Silver Records :", silver_drivers_df.count())
 
print("Silver Drivers data successfully written to:")
print(silver_path)